In [4]:
# pip install pypdf

from io import BytesIO
from pypdf import PdfReader
import re
from typing import Dict, Optional

FIELD_PATTERNS = {
    "landlord_name": r"(?:Landlord Name|Lessor)\s*:\s*([A-Za-z .]+)",
    "tenant_name":   r"(?:Tenant Name|Lessee)\s*:\s*([A-Za-z .]+)",
    "tenant_address":r"(?:Tenant Address|Address)\s*:\s*([^\n]+)",
    "rent_amount":   r"(?:Monthly Rent|Rent)\s*:\s*₹?\s*([\d,]+)",
    "start_date":    r"(?:Start Date|Commencement)\s*:\s*([0-9]{1,2}[-/][0-9A-Za-z]{2,9}[-/][0-9]{2,4})",
    "end_date":      r"(?:End Date|Expiry)\s*:\s*([0-9]{1,2}[-/][0-9A-Za-z]{2,9}[-/][0-9]{2,4})",
    "aadhar":        r"(?:Aadhaar|Aadhar)\D*([\d ]{12,14})",
    "pan":           r"(?:PAN)\D*([A-Z]{5}\d{4}[A-Z])",
}

def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    """Extracts text using the PDF text layer (fast, no OCR)."""
    reader = PdfReader(BytesIO(pdf_bytes))
    parts = []
    for p in reader.pages:
        t = p.extract_text() or ""
        parts.append(t)
    return "\n".join(parts)

def _find(pattern: str, text: str) -> Optional[str]:
    m = re.search(pattern, text, flags=re.IGNORECASE)
    return m.group(1).strip() if m else None

def extract_lease_details(pdf_bytes: bytes) -> Dict[str, Optional[str]]:
    """
    Returns a dict of extracted fields from a lease PDF.
    Designed to work out-of-the-box with your demo_lease.pdf.
    """
    text = extract_text_from_pdf(pdf_bytes)

    data = {k: _find(pat, text) for k, pat in FIELD_PATTERNS.items()}

    # Normalize some values
    if data.get("rent_amount"):
        data["rent_amount"] = data["rent_amount"].replace(",", "")
    if data.get("aadhar"):
        data["aadhar"] = data["aadhar"].replace(" ", "")

    return {
        "landlord_name": data.get("landlord_name"),
        "tenant_name": data.get("tenant_name"),
        "tenant_address": data.get("tenant_address"),
        "rent_amount_inr": data.get("rent_amount"),
        "start_date": data.get("start_date"),
        "end_date": data.get("end_date"),
        "aadhar": data.get("aadhar"),
        "pan": data.get("pan"),
        "raw_preview": text[:1000],  # optional: helps debugging
    }

# --- Example usage ---
with open("/Users/anshagarwal/Desktop/KirayaEase/data/demo_lease.pdf", "rb") as f:
    pdf_bytes = f.read()
details = extract_lease_details(pdf_bytes)
print(details)

{'landlord_name': 'John Smith', 'tenant_name': 'Ansh Agarwal', 'tenant_address': '123 Palm Street, Pune, MH', 'rent_amount_inr': None, 'start_date': '01-09-2025', 'end_date': '31-08-2026', 'aadhar': '123456789123', 'pan': 'ABCDE1234F', 'raw_preview': 'Residential Lease Agreement\nLandlord Name: John Smith\nAddress: 123 Palm Street, Pune, MH\nTenant Name: Ansh Agarwal\nTenant Address: 45 Maple Ave, Bengaluru, KA\nMonthly Rent: ■12,500\nStart Date: 01-09-2025\nEnd Date: 31-08-2026\nTenant Aadhar: 1234 5678 9123\nTenant PAN: ABCDE1234F\nThis is a demo lease PDF for KirayaEase.\n'}


{'landlord_name': 'John Smith',
 'tenant_name': 'Ansh Agarwal',
 'tenant_address': '123 Palm Street, Pune, MH',
 'rent_amount_inr': None,
 'start_date': '01-09-2025',
 'end_date': '31-08-2026',
 'aadhar': '123456789123',
 'pan': 'ABCDE1234F',
 'raw_preview': 'Residential Lease Agreement\nLandlord Name: John Smith\nAddress: 123 Palm Street, Pune, MH\nTenant Name: Ansh Agarwal\nTenant Address: 45 Maple Ave, Bengaluru, KA\nMonthly Rent: ■12,500\nStart Date: 01-09-2025\nEnd Date: 31-08-2026\nTenant Aadhar: 1234 5678 9123\nTenant PAN: ABCDE1234F\nThis is a demo lease PDF for KirayaEase.\n'}